# Øvelse: Tweets som data

**Social Data Science 1: lektion 1 til 3**

Nu samler vi det hele. I skal indlæse et rigtigt datasæt, rense teksten, tælle
ord, koble to kilder sammen og vende en tabel på hovedet.

Datasættet er alle tweets fra kontoen `@realDonaldTrump` i 2016. Ca. 4.000
tekster, hentet af to forskellige personer på to forskellige tidspunkter (se https://www.kaggle.com/datasets/austinreese/trump-tweets). 

**Det vigtige er teknikken, ikke resultaterne.** I skal ikke konkludere noget om
amerikansk politik i dag. I skal kunne indlæse, rense, koble og omforme data.

**Tre filer:**

| Fil | Indhold |
|---|---|
| `tweets_2016_kilde_a.csv` | id, dato, tekst, retweets, favoritter, hashtags, mentions |
| `tweets_2016_kilde_b.csv` | id, dato, retweets, favoritter, geo |
| `maaneder.csv` | opslagstabel: månedsnummer, navn og fase i valgkampen |



---

## Del 1: Indlæsning og overblik

### 1.1 Læs filen ind

**Trin 1: kør cellen.** `read_csv()` læser en kommasepareret fil ind som en tabel.


In [ ]:
import pandas as pd

tweets = pd.read_csv("data_tweets_2016_kilde_a.csv")

print(tweets.head())


**Trin 2: hvor stort er det?**

`.shape` giver `(rækker, kolonner)`. `.columns` giver kolonnenavnene.

Hvor mange tweets er der? Og hvor mange oplysninger har vi om hver enkelt?


In [ ]:
# Din kode her



**Trin 3: hvilke typer?**

Kør cellen og kig godt på `date`.

`.dtypes` viser, hvilken **type** pandas har givet hver kolonne. Typen bestemmer,
hvad I kan gøre med kolonnen: `int64` og `float64` kan der regnes på, `str` (eller
`object`) er tekst, `bool` er sandt/falsk.

pandas gætter typen ud fra indholdet, da filen blev læst ind. Som regel gætter den
rigtigt, men den siger ikke til, når den gætter forkert.


In [ ]:
print(tweets.dtypes)


Kolonnen `date` ser ud som en dato, men pandas har læst den som **tekst**
(`object` eller `string`). Det er ikke en fejl, pandas gætter ikke på datoer,
medmindre man beder om det.

Det betyder, at `tweets["date"] > "2016-06-01"` sammenligner tekst, ikke tid.
Her går det tilfældigvis godt, fordi formatet `ÅÅÅÅ-MM-DD` sorterer ens som tekst
og som dato. Med formatet `01-06-2016` ville det gå galt.

Vi retter det senere. Læg mærke til det nu.


### 1.2 Se på én enkelt række

`.loc` henter bestemte rækker og kolonner ud af en tabel. Den tager to ting adskilt
af komma: **række** først, så **kolonne**.

    tweets.loc[0]               # hele række 0
    tweets.loc[0, "content"]    # kun feltet 'content' i række 0

Rækketallet er ikke en position, men den **etiket**, rækken har i indekset. Her er
de det samme, fordi indekset bare er 0, 1, 2, 3. Men efter en filtrering eller en
sortering følger etiketterne med. Så kan `.loc[0]` sagtens være den femte række
nedefra, og `.loc[3]` findes måske slet ikke.

Print teksten fra tweet nummer 0, og derefter fra tweet nummer 100.


In [ ]:
print(tweets.loc[0, "content"])

# Din kode her: tweet nummer 100


---

## Del 2: Én tweet ad gangen

Her er `rens()` fra lektion 2, udvidet med et par flere tegn. Kør cellen.


In [ ]:
def rens(t):
    t = str(t).lower()
    for tegn in [".", ",", "!", "?", ":", ";", '"']:
        t = t.replace(tegn, "")
    return t.split()

print(rens("Er DET her? Ja, det er."))


Bemærk `str(t)` på første linje. Nogle felter i et rigtigt datasæt er tomme, og
et tomt felt er ikke en tekst, det er værdien `NaN`. `str()` gør det om til noget,
`.lower()` kan arbejde med, i stedet for at koden stopper midt i en løkke.


### 2.1 Rens den første tweet

Kør `rens()` på teksten fra tweet nummer 0. Hvor mange ord er der?

Kig på listen. Er der noget, der ikke rigtig er et ord?


In [ ]:
# Din kode her



---

## Del 3: Tæl ord på tværs af alle tweets

### 3.1 Byg optællingen

**Trin 1: først kun de ti første tweets.**

Start småt, så I kan se, om det virker. `tweets["content"].head(10)` giver de
første ti tekster.

Udfyld den ene plads. `.get(o, 0)` betyder: *giv mig værdien for ordet `o`, og
hvis ordet ikke findes endnu, så giv mig 0 i stedet.* Uden nullet ville der ikke
være noget at lægge 1 til, første gang I møder et ord.


In [ ]:
antal = {}

for t in tweets["content"].head(10):
    for o in rens(t):
        antal[o] = ______ + 1

print(len(antal))


**Trin 2: nu alle sammen.**

Fjern `.head(10)`, så løkken kører over alle tweets. Husk at nulstille `antal`
først, ellers lægges de ti første oveni igen.

Hvor mange **unikke** ord er der? Og hvor mange ord i alt?

*Hint:* `sum(antal.values())` giver totalen.


In [ ]:
# Din kode her



### 3.2 De hyppigste ord

Byg en liste af `(antal, ord)`-tupler, sortér faldende, og print de ti øverste.

Tre ting bruges:

- **`.items()`** giver nøgle og værdi sammen, ét par ad gangen.
- **`.append()`** lægger ét element til sidst i listen. Bemærk de dobbelte parenteser: de indre laver tuplen, de ydre er selve kaldet.
- **`sort()`** sorterer på stedet og returnerer `None`. Skriv `par.sort()`, ikke `par = par.sort()`. `reverse=True` vender rækkefølgen.

In [ ]:
# Din kode her



### 3.3 Til diskussion

Kig på jeres top 10.

1. Hvor mange af de ti ord siger noget om indholdet?
2. To af posterne er `@` og `#` helt alene. Hvordan kan et enkelt tegn blive til et
   ord? 
3. Hvad ville I gøre ved det? Skriv én sætning om, hvad `rens()` mangler.



---

## Del 4: Hashtags

Kolonnen `hashtags` har enten en tekst eller ingenting. Har en tweet flere hashtags,
står de i samme felt adskilt af komma.

**Trin 1: kig på formen.** Kør cellen.


In [ ]:
print(tweets["hashtags"].head(8))
print()
print(tweets["hashtags"].notna().sum(), "tweets har mindst ét hashtag")


### 4.1 Tæl hashtags

Byg en dictionary `htal` med antal gange, hvert hashtag optræder.

To ting adskiller den fra ordoptællingen:

- De tomme felter skal **springes over**. Brug `if pd.isna(h): continue`.
- Feltet skal deles på komma, ikke på mellemrum: `h.split(",")`.

```python
htal = {}

for h in tweets["hashtags"]:
    if pd.isna(h):
        continue
    for tag in str(h).split(","):
        # tæl op her
```

Hvor mange unikke hashtags er der? Hvad er de fem hyppigste?


In [ ]:
# Din kode her



### 4.2 Til diskussion

1. Hvad ville der ske, hvis I fjernede `if pd.isna(h): continue`? Prøv det.
2. Antallet af tweets med hashtag er mindre end antallet af hashtags i alt.
   Hvorfor?
3. `#Trump2016` og `#Trump2016pic` tælles som to forskellige hashtags. Er det
   rigtigt? Kig på den rå tekst i et par af de tweets.


---

## Del 5: Filtrering med booleans

En sammenligning på en hel kolonne giver en boolean per række.

**Trin 1: kør cellen og kig på hvad der kommer ud.**


In [ ]:
stor = tweets["retweets"] > 10000

print(stor.head())
print()
print(stor.sum(), "tweets har over 10.000 retweets")


### 5.1 Brug den til at filtrere

`tweets[stor]` giver kun de rækker, hvor værdien er `True`.

1. Hvor mange rækker har `tweets[stor]`? Stemmer det med `stor.sum()`?
2. Hvad er den gennemsnitlige antal favoritter blandt de store tweets? Og blandt
   alle tweets?
3. Byg en boolean, der er sand, når en tweet har **både** over 10.000 retweets
   **og** mindst ét hashtag. Husk `&` i stedet for `and`, når det er hele kolonner.

*Hint til 3:* `tweets["hashtags"].notna()` giver den anden halvdel.


In [ ]:
# Din kode her



---

## Del 6: To kilder, samme tweets

De samme tweets er hentet to gange, af to forskellige personer. Nu skal de kobles.

**Trin 1: læs kilde B ind og sammenlign kolonnerne.**


In [ ]:
kilde_b = pd.read_csv("data_tweets_2016_kilde_b.csv")

print(kilde_b.head())
print()
print(list(tweets.columns))
print(list(kilde_b.columns))


### 6.1 Hvor meget overlapper de?

**Trin 2: tæl først, kobl bagefter.**

`set()` fjerner gengangere, så I får de id'er, der faktisk findes. `-` mellem to
sets giver det, der er i den ene men ikke i den anden.

In [ ]:
a_ider = set(tweets["id"])
b_ider = set(kilde_b["id"])

print("i begge:  ", len(a_ider & b_ider))
print("kun i A:  ", len(a_ider - b_ider))
print("kun i B:  ", len(b_ider - a_ider))


### 6.2 Kobl dem

**Gæt først.** Skriv de fire tal ned:

| `how=` | Mit gæt |
|---|---|
| `"inner"` | |
| `"left"` | |
| `"right"` | |
| `"outer"` | |

Kør derefter cellen.

`suffixes` bestemmer, hvad de kolonner skal hedde, som findes i **begge** tabeller
— her `retweets` og `favorites` og `date`.


In [ ]:
for h in ["inner", "left", "right", "outer"]:
    k = pd.merge(tweets, kilde_b, on="id", how=h, suffixes=("_a", "_b"))
    print(h, len(k))


### 6.3 Sammenlign de to kilder

Lav en inner join og gem den som `samlet`. Undersøg så:

1. Hvor stor en andel af rækkerne har samme `retweets` i de to kilder?
   *Hint:* `(samlet["retweets_a"] == samlet["retweets_b"]).mean()`
2. Er `date_a` og `date_b` ens? Kig på de første fem rækker af begge.
3. Hvad indeholder kolonnen `geo`? Prøv `samlet["geo"].notna().sum()`.


In [ ]:
# Din kode her



### 6.4 Til diskussion

1. Retweet-tallene er forskellige for næsten alle tweets. Er den ene kilde forkert?
2. Datoerne er forskudt med et fast antal timer, men ikke det samme antal hele
   året. Hvad kan forklare det?
3. Kolonnen `geo` er tom for alle rækker. Skal den slettes, eller skal den blive?
   Hvad fortæller en tom kolonne jer?
4. Hvis I skulle skrive i en metodesektion, hvilken af de to kilder ville I bruge
   til retweet-tal, og hvad ville I skrive om det?


---

## Del 7: Måneder og form

### 7.1 Træk måneden ud

`date` er tekst. For at få måneden ud skal den først laves om til en rigtig dato.

Kør cellen.


In [ ]:
tweets["dato"] = pd.to_datetime(tweets["date"])
tweets["maaned"] = tweets["dato"].dt.month

print(tweets[["date", "dato", "maaned"]].head())
print(tweets["maaned"].value_counts().sort_index())

### 7.2 Kobl med opslagstabellen

**Trin 1. læs den ind og kig.** Hvor mange måneder er der?

In [ ]:
maaneder = pd.read_csv("maaneder.csv")

print(maaneder)

**Trin 2: kobl.**

Gæt antal rækker for `how="inner"` og `how="left"`, før I kører.

Kør derefter begge, og find ud af: hvilke tweets forsvinder ved en inner join, og
hvor mange er der af dem?

In [ ]:
# Din kode her

Det er den vigtigste enkeltstående ting i hele notebooken. Opslagstabellen mangler
én måned, og en inner join smider alle tweets fra den måned væk, uden en advarsel.

I et rigtigt projekt ville I ikke opdage det, medmindre I talte rækker.

### 7.3 Fra langt til bredt

Vi vil tælle tre valgte ord per måned.

**Trin 1: én måned først.** Kør cellen og se, hvordan man skærer en delmængde ud.

In [ ]:
januar = tweets[tweets["maaned"] == 1]
print(len(januar))

tael = {}
for t in januar["content"]:
    for o in rens(t):
        tael[o] = tael.get(o, 0) + 1

print(tael.get("great", 0))

**Trin 2: alle måneder, alle tre ord.**

Byg en liste af dictionaries med tre nøgler: `maaned`, `ord` og `antal`. Send den
gennem `pd.DataFrame()`.

```python
ord_valg = ["great", "thank", "america"]
raekker = []

for m in sorted(tweets["maaned"].unique()):
    # skær måneden ud
    # tæl ordene i den måned
    for w in ord_valg:
        raekker.append({"maaned": m, "ord": w, "antal": ______})
```

I skal ende med 36 rækker: 12 måneder gange 3 ord. Det er **langt format**.

In [ ]:
# Din kode her

**Trin 3: vend den.**

Brug `pivot()` til at få ordene i rækkerne og månederne i kolonnerne.

Hvor mange rækker og kolonner får I nu? Og hvilket format er lettest at læse?

In [ ]:
# Din kode her

**Trin 4: og tilbage igen.**

Brug `reset_index()` og `melt()` til at komme tilbage til de 36 rækker.

Hvorfor er `reset_index()` nødvendig først?


In [ ]:
# Din kode her